In [ ]:

from helpers.dialect_classifier import build_classifier, evaluate_dialect

clf = build_classifier()          # trains once
scores = evaluate_dialect("Sunshine Coast students push back as Australia positions itself as a global AI leader Tue 23 Jun 2026 at 2:53pm In short: Sunshine Coast Council has partnered with Google and NEXTDC to build an AI data centre in Maroochydore. High school and university students have banded together to protest the development. What's next? Council is looking to capitalise on Maroochydore's proximity to the undersea cable to grow the CBD into a digital hub. A $200 million data centre under construction on the Sunshine Coast has ignited a protest movement from students concerned about its environmental and social impacts. The five-storey NEXTDC SC2 data centre under construction in Maroochydore city centre will become one of 162 data centres operating across Australia. At least 90 more are planned nationwide, prompting a group of Sunshine Coast residents to call for stronger regulations as the infrastructure expands into suburban areas. An AI data centre is a facility that houses the specific IT infrastructure needed to train, deploy and deliver AI applications and services. Data centre infrastructure uses significant volumes of water to keep machines cool. The Greens are calling for a moratorium on \"hyperscale\" data centre development approvals, arguing the \"energy vampires\" are putting pressure on power supplies, water resources and communities. It is unknown how many resources the Maroochydore AI site will use. The Climate Council of Australia said in 2024-25 data centres used about 4 terawatt-hours (TWh), or 2 per cent of the electricity in Australia's main grid, the National Electricity Market (NEM). This is equivalent to the electricity use of more than 700,000 homes. Loading...At last week's Protect Sunshine Coast rally, organiser Ruby Dyer, 16, said her concerns extended beyond the Maroochydore facility to broader anxieties about the increasing role of AI in everyday life. \"We're wanting to show our leaders that there is a large group of people who aren't OK with the lack of regulations to hopefully put pressure on them to get these regulations put in place,\" she said. Ruby said their concerns related to the environmental impacts, misuse by \"bad users\", and the diminishing prospects for young people as they entered the workforce. \"There are a lot of jobs in so many different professions that are being affected, [\u2026] lost or replaced,\\\"she said. \"AI is doing it for them, so there is a big concern with there not being as many jobs when I go into the workforce.\" University student Kate McGeechan, 19, was among the dozen young people who ditched studies and attended last week's strike outside the Sunshine Coast Council building in Maroochydore. She was motivated after seeing Ruby's Instagram and believed young Australians had valid concerns about society's growing dependence on the technology. \"I think it was really admirable to be organised by these high school students,\" Ms McGeechan said. She said she could acknowledge the human desire to want to automate processes, but she wanted people to carefully think about becoming reliant. \"I want people to think twice before using AI for their everyday activities, and just to remember that, at the end of the day, we're all human and we're all capable,\" she said. She also questioned how increasing dependence could shape future professions. \"Are we going to have a new generation of lawyers and doctors who might not be entirely qualified to make the judgements that they're supposed to make on a daily basis?\"Ms McGeechan said. The students are planning more action, including a social media campaign to encourage more young people to investigate regulations around AI data centres. University of the Sunshine Coast discipline lead of technology Erica Mealy said some concerns raised by the students mirrored legitimate debates occurring nationally and internationally. However, Dr Mealy said AI had the potential to improve lives when used appropriately, especially when it came to dirty, dangerous and undesirable work. \"Leave humans the creativity, leave humans the great thinking and the research advances and take away some of the stuff that's really irritating for us,\"she said. Dr Mealy said the proposed Sunshine Coast facility could deliver significant benefits to the region and rural Queensland, such as advances in remote health care. Queensland's first regional data centre was built in Maroochydore in 2021, which is close to the construction site of the new data centre. In a statement to the ABC, a Sunshine Coast Council representative said partnerships with NEXTDC and Google were designed to secure long-term economic growth and digital resilience for the region. They said the key benefits would be more reliable connectivity for cloud services, data transfer and digital operations, including supporting modern AI systems. The representative said the SC2 facility would be carbon neutral but did not answer specific questions from ABC on environmental assessments.")
predicted = max(scores, key=scores.get)   # e.g. "UK"
print('Predicted:', predicted)


In [9]:
predicted

'AUS'

## Article Loader

Walk `./articles/` and collect every individual article JSON with its metadata.  
Uses `OUTLET_TO_DIALECT` and `_load_individual_file` from `dialect_classifier.py` so the dialect mapping stays consistent with the training pipeline.

In [10]:
import os
import json
from collections import Counter, defaultdict
from helpers.dialect_classifier import OUTLET_TO_DIALECT, _load_individual_file

def load_articles(articles_dir: str = "./articles") -> list[dict]:
    """
    Walk *articles_dir* and return one record per article JSON.
    Skips files with empty text and bundle-style .json files at the outlet level.

    Returns a list of dicts:
        full_text, dialect, outlet, topic, title
    """
    records = []
    for topic in sorted(os.listdir(articles_dir)):
        topic_path = os.path.join(articles_dir, topic)
        if not os.path.isdir(topic_path):
            continue
        for outlet in sorted(os.listdir(topic_path)):
            outlet_path = os.path.join(topic_path, outlet)
            if not os.path.isdir(outlet_path):
                continue                          # skip bundle .json files
            dialect = OUTLET_TO_DIALECT.get(outlet)
            if dialect is None:
                continue
            for fname in sorted(os.listdir(outlet_path)):
                if not fname.endswith(".json"):
                    continue
                fpath = os.path.join(outlet_path, fname)
                try:
                    with open(fpath, encoding="utf-8") as f:
                        raw = json.load(f)
                    full_text = _load_individual_file(fpath).strip()
                    if not full_text:
                        continue
                    records.append({
                        "full_text": full_text,
                        "dialect":   dialect,
                        "outlet":    outlet,
                        "topic":     topic,
                        "title":     raw.get("title", ""),
                    })
                except (json.JSONDecodeError, OSError):
                    continue
    return records


all_articles = load_articles()

# ── Summary table ────────────────────────────────────────────────────────────
topic_counts = Counter(a["topic"] for a in all_articles)
outlet_counts = Counter((a["outlet"], a["dialect"]) for a in all_articles)

print(f"Total articles loaded: {len(all_articles)}\n")
print(f"{'Outlet':<12}  {'Dialect':<7}  {'Count':>5}")
print("-" * 30)
for (outlet, dialect), n in sorted(outlet_counts.items()):
    print(f"{outlet:<12}  {dialect:<7}  {n:>5}")
print()
print(f"{'Topic':<30}  {'Count':>5}")
print("-" * 38)
for topic, n in sorted(topic_counts.items()):
    print(f"{topic:<30}  {n:>5}")

Total articles loaded: 239

Outlet        Dialect  Count
------------------------------
abc_au        AUS         80
abc_us        US          79
bbc           UK          80

Topic                           Count
--------------------------------------
artificial_intelligence            30
climate_change                     30
elections                          30
energy                             30
global_economy                     30
immigration                        30
public_health                      30
ukraine_war                        29


## Sanity Check — Classifier Accuracy on All Region-Specific Articles

Run `clf.evaluate_dialect()` on every article and check whether the predicted dialect matches the article's true outlet region.  

> **Note:** This tags all articles with spaCy — expect ~1–2 minutes for ~240 articles.

**Expected:** accuracy well above random (33%) for each dialect if the POS n-gram model is working. US and AU/UK tend to be easier to separate than AU vs UK.

In [11]:
from helpers.dialect_classifier import scores_to_probs

DIALECTS = ["US", "UK", "AUS"]

# ── Run classifier on every article ─────────────────────────────────────────
predictions = []   # (true_dialect, predicted_dialect, title, topic, probs)

for i, art in enumerate(all_articles):
    log_scores = clf.evaluate_dialect(art["full_text"])
    probs      = scores_to_probs(log_scores)
    predicted  = max(log_scores, key=log_scores.get)
    predictions.append((art["dialect"], predicted, art["title"], art["topic"], probs))
    if (i + 1) % 25 == 0:
        print(f"  classified {i+1}/{len(all_articles)} …")

print(f"Done. {len(predictions)} articles classified.\n")

# ── Per-dialect accuracy ──────────────────────────────────────────────────────
correct_by_dialect = defaultdict(int)
total_by_dialect   = defaultdict(int)

for true, pred, *_ in predictions:
    total_by_dialect[true] += 1
    if true == pred:
        correct_by_dialect[true] += 1

overall_correct = sum(correct_by_dialect.values())
print(f"Overall accuracy: {overall_correct}/{len(predictions)}  "
      f"({overall_correct/len(predictions):.1%})\n")

print(f"{'Dialect':<8}  {'Correct':>7}  {'Total':>6}  {'Accuracy':>9}")
print("-" * 36)
for d in DIALECTS:
    n   = total_by_dialect[d]
    cor = correct_by_dialect[d]
    print(f"{d:<8}  {cor:>7}  {n:>6}  {cor/n:>8.1%}")

# ── Confusion matrix ──────────────────────────────────────────────────────────
print("\nConfusion matrix  (rows = true, cols = predicted):")
print(f"{'':>8}  " + "  ".join(f"{d:>5}" for d in DIALECTS))
print("-" * (8 + 9 * len(DIALECTS)))

confusion = defaultdict(Counter)
for true, pred, *_ in predictions:
    confusion[true][pred] += 1

for true in DIALECTS:
    row = "  ".join(f"{confusion[true][pred]:>5}" for pred in DIALECTS)
    print(f"{true:>8}  {row}")

# ── Sample misclassifications ─────────────────────────────────────────────────
print("\nSample misclassified articles (up to 2 per dialect):")
shown = defaultdict(int)
for true, pred, title, topic, probs in predictions:
    if true != pred and shown[true] < 2:
        prob_str = "  ".join(f"{d}={probs[d]:.2%}" for d in DIALECTS)
        print(f"\n  [true={true}  pred={pred}]  {topic} | {title[:55]}")
        print(f"    probs: {prob_str}")
        shown[true] += 1

  classified 25/239 …
  classified 50/239 …
  classified 75/239 …
  classified 100/239 …
  classified 125/239 …
  classified 150/239 …
  classified 175/239 …
  classified 200/239 …
  classified 225/239 …
Done. 239 articles classified.

Overall accuracy: 233/239  (97.5%)

Dialect   Correct   Total   Accuracy
------------------------------------
US             79      79    100.0%
UK             75      80     93.8%
AUS            79      80     98.8%

Confusion matrix  (rows = true, cols = predicted):
             US     UK    AUS
-----------------------------------
      US     79      0      0
      UK      5     75      0
     AUS      1      0     79

Sample misclassified articles (up to 2 per dialect):

  [true=UK  pred=US]  artificial_intelligence | Anthropic: AI could escape human control
    probs: US=98.04%  UK=0.78%  AUS=1.18%

  [true=UK  pred=US]  artificial_intelligence | BBC Inside Science
    probs: US=87.98%  UK=12.02%  AUS=0.00%

  [true=AUS  pred=US]  ukraine_war | Ukr

## Spelling Markers Check (`spelling_markers.py`)

Verify that `compute_lsr` and `find_switches` from `spelling_markers.py` produce sensible results on the real article corpus.

**Expected:**
- `ap_us` articles → mean LSR near **1.0** (mostly US spelling markers)
- `bbc` / `abc_au` articles → mean LSR near **0.0** (AU/UK spellings dominate)

`find_switches` demo: for a true AU article, expecting `AU` should find *few* switches; expecting `US` should surface *many* — confirming the tool would correctly flag LLM dialectal drift.

In [ ]:
from helpers.spelling_markers import load_spelling_markers, compute_lsr, find_switches, strip_quotes

markers = load_spelling_markers()
print(f"Spelling markers loaded — US: {len(markers['US']):,}  UK: {len(markers['UK']):,}  AU: {len(markers['AU']):,}\n")

# dialect_classifier uses "AUS"; spelling_markers uses "AU"
DIALECT_MAP = {"US": "US", "UK": "UK", "AUS": "AU"}

# ── LSR for every article ─────────────────────────────────────────────────────
lsr_by_outlet: dict[str, list[float]] = defaultdict(list)

for art in all_articles:
    clean  = strip_quotes(art["full_text"])
    result = compute_lsr(clean, markers)
    if result["total_dialect_tokens"] > 0:
        lsr_by_outlet[art["outlet"]].append(result["lsr"])

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"{'Outlet':<12}  {'Dialect':<7}  {'N':>4}  {'Mean LSR':>9}  {'Min':>6}  {'Max':>6}")
print("=" * 52)
for outlet in ("ap_us", "bbc", "abc_au"):
    dialect = OUTLET_TO_DIALECT[outlet]
    lsrs    = lsr_by_outlet[outlet]
    if lsrs:
        print(f"{outlet:<12}  {dialect:<7}  {len(lsrs):>4}  "
              f"{sum(lsrs)/len(lsrs):>9.4f}  {min(lsrs):>6.4f}  {max(lsrs):>6.4f}")
    else:
        print(f"{outlet:<12}  {dialect:<7}  {'—':>4}  {'n/a':>9}")

# ── find_switches demo ────────────────────────────────────────────────────────
print("\n" + "=" * 64)
print("find_switches demo — one AU article, two expected-dialect views")
print("=" * 64)

au_sample = next(a for a in all_articles if a["outlet"] == "abc_au")
clean_au  = strip_quotes(au_sample["full_text"])

print(f"\nArticle: {au_sample['title'][:70]}")
print(f"Topic  : {au_sample['topic']}\n")

for expected in ("AU", "US"):
    switches = find_switches(clean_au, expected, markers)
    label = ("✓ own dialect — few switches expected"
             if expected == "AU" else
             "✗ wrong dialect — many switches expected (simulates LLM drift)")
    print(f"  expected_dialect='{expected}'  →  {len(switches)} switch(es)  [{label}]")
    for sw in switches[:6]:
        print(f"    '{sw['token']}'  ({sw['switch']})  →  equiv: '{sw['equivalent']}'")
    if len(switches) > 6:
        print(f"    … and {len(switches) - 6} more")